In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP03 - TP Services
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Did the unit use third party intermediaries or subcontractors during the assessment 
   period to perform marketing, business development, sales, consulting, or brokerage?
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the new Master AU List + TP Unit Mapping, strictly 
      filtered by 'Jurisdiction' for Canada, Hong Kong, and Barbados.
   2. RISK FLAG LOGIC: Flags engagements as high risk if they meet either condition:
      - KPI_Number = '8.5' AND Value = '100'
      - primary_product_serv matches one of the 14 predefined marketing/printing categories.
   3. TPRM STRING MAPPING: Maps flagged TP engagements to AU IDs by checking if the 
      'TPRM_Units' string from the mapping table exists inside the text of the 
      'owning_lob' or 'lob_using' columns.
   4. FINAL OUTPUT: If an mapped AU has 1 or more flagged engagements, outputs 'Yes', 
      otherwise safely defaults to 'No'. Maintains strict 6-column structure.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data and strictly filter by target jurisdictions
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment 
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
),

Mapped_AUs AS (
    -- 2. Grab every AU that currently has TPRM units mapped to it
    SELECT DISTINCT TRIM(CAST(Assessable_Unit_ID AS STRING)) AS AU_ID 
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
),

All_Base_AUs AS (
    -- 3. Combine filtered Master List and Mapping to create the ultimate base table
    SELECT 
        COALESCE(mast.BusinessID, map.AU_ID) AS Base_AU_ID,
        mast.AU_Name,
        mast.Business_Segment,
        CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_ABAC_AU_List
    FROM Master_AUs mast
    FULL OUTER JOIN Mapped_AUs map 
        ON mast.BusinessID = map.AU_ID
),

Flagged_Engagements AS (
    -- 4. Filter the TP data based on the KPI rules OR the specific service categories
    SELECT DISTINCT
        EngagementNumber,
        owning_lob,
        lob_using
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
      AND (
          -- Condition 1: KPI 8.5 equals 100
          (TRIM(KPI_Number) = '8.5' AND TRIM(Value) = '100')
          
          OR 
          
          -- Condition 2: Matches target Marketing/Printing services (Col F)
          TRIM(primary_product_serv) IN (
              'Marketing media services',
              'Marketing and distribution',
              'Direct marketing print service',
              'Print advertising',
              'Printing',
              'Management and Business Professionals and Administrative Services',
              'Marketing analysis',
              'Publicity and marketing support services',
              'Marketing communications agencies',
              'Advertising agency services',
              'Business forms or questionnaires',
              'Specialized warehousing and storage',
              'Stationery or business form printing',
              'Published Products'
          )
      )
),

Engagements_By_AU AS (
    -- 5. Map the flagged engagements using the TPRM unit mapping table
    SELECT 
        TRIM(CAST(map.Assessable_Unit_ID AS STRING)) AS Mapped_AU_ID,
        COUNT(DISTINCT f.EngagementNumber) AS Match_Count
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping map
    -- Joins if the mapped TPRM string is found inside owning_lob or lob_using
    INNER JOIN Flagged_Engagements f
        ON UPPER(f.owning_lob) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
        OR UPPER(f.lob_using) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
    WHERE map.Assessable_Unit_ID IS NOT NULL
    GROUP BY TRIM(CAST(map.Assessable_Unit_ID AS STRING))
)

-- 6. Final Template: Strict 6-column output
SELECT 
    a.Base_AU_ID AS BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP03' AS QuestionID,               
    
    -- Yes/No Logic based on the match count
    CASE 
        WHEN COALESCE(e.Match_Count, 0) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response,
    
    a.In_ABAC_AU_List
    
FROM All_Base_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.Base_AU_ID = e.Mapped_AU_ID
ORDER BY a.Base_AU_ID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP03 - TP Services Drill-Down
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   PURPOSE: 
   Provides a row-by-row view of every Third Party Engagement mapped to an AU that 
   triggered a 'Yes' for TP03. Displays the mapping validation strings alongside the 
   raw KPI variables and the primary product service to verify exactly which condition 
   caught the engagement.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
),

Mapped_AUs AS (
    SELECT DISTINCT TRIM(CAST(Assessable_Unit_ID AS STRING)) AS AU_ID 
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
),

All_Base_AUs AS (
    SELECT 
        COALESCE(mast.BusinessID, map.AU_ID) AS Base_AU_ID,
        mast.AU_Name,
        CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_ABAC_AU_List
    FROM Master_AUs mast
    FULL OUTER JOIN Mapped_AUs map 
        ON mast.BusinessID = map.AU_ID
),

Flagged_Engagements AS (
    SELECT 
        EngagementNumber,
        ThirdPartyName,
        owning_lob,
        lob_using,
        KPI_Number,
        Value,
        primary_product_serv,
        CASE 
            WHEN TRIM(KPI_Number) = '8.5' AND TRIM(Value) = '100' THEN 'Flagged via KPI 8.5'
            ELSE 'Flagged via Target Service Category'
        END AS Flag_Reason
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
      AND (
          (TRIM(KPI_Number) = '8.5' AND TRIM(Value) = '100')
          OR 
          TRIM(primary_product_serv) IN (
              'Marketing media services', 'Marketing and distribution', 'Direct marketing print service',
              'Print advertising', 'Printing', 'Management and Business Professionals and Administrative Services',
              'Marketing analysis', 'Publicity and marketing support services', 'Marketing communications agencies',
              'Advertising agency services', 'Business forms or questionnaires', 'Specialized warehousing and storage',
              'Stationery or business form printing', 'Published Products'
          )
      )
)

SELECT DISTINCT
    a.Base_AU_ID AS BusinessID,
    a.AU_Name,
    a.In_ABAC_AU_List,
    f.EngagementNumber,
    f.ThirdPartyName,
    map.TPRM_Units AS Successfully_Mapped_String,
    f.owning_lob AS Raw_Col_K_owning_lob,
    f.lob_using AS Raw_Col_L_lob_using,
    f.KPI_Number,
    f.Value,
    f.primary_product_serv AS Col_F_Category,
    f.Flag_Reason
    
FROM All_Base_AUs a

-- Inner join AU list to mapping table
INNER JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map
    ON a.Base_AU_ID = TRIM(CAST(map.Assessable_Unit_ID AS STRING))

-- Inner join mapping string to raw TP data via LIKE
INNER JOIN Flagged_Engagements f
    ON UPPER(f.owning_lob) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
    OR UPPER(f.lob_using) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
    
ORDER BY BusinessID;